In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Load the data generated from the geolocation step
df = pd.read_csv('../data/processed/ecommerce_cleaned_geo.csv')
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/ecommerce_cleaned_geo.csv'

In [ ]:
print("Engineering behavioral and temporal features...")

# 1. Time-Since-Signup (Converted to hours as a float)
# Fraud Pattern: Bots buy things immediately after automating account creation.
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 3600.0

# 2. Temporal Features
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek

# 3. Device Shared Velocity
# Count how many unique users are sharing the exact same device ID
df['user_count_per_device'] = df.groupby('device_id')['user_id'].transform('count')

# 4. IP Shared Velocity
# Count how many unique users are sharing the exact same IP Address
df['user_count_per_ip'] = df.groupby('ip_address')['user_id'].transform('count')

print("New behavioral features successfully created.")

In [ ]:
# Drop unique IDs and raw timestamps that models can't generalize from
columns_to_drop = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address']
df_ml = df.drop(columns=columns_to_drop)

# Handle high-cardinality categorical features: Top 15 countries, group rest as 'Other'
top_countries = df_ml['country'].value_counts().index[:15]
df_ml['country'] = df_ml['country'].apply(lambda x: x if x in top_countries else 'Other')

# Perform One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df_ml, columns=['source', 'browser', 'sex', 'country'], drop_first=True)

# Separate independent features (X) from the target label (y)
X = df_encoded.drop(columns=['class'])
y = df_encoded['class']

In [ ]:
# Split data into 80% Training and 20% Testing sets
# stratify=y preserves the exact fraud-to-legitimate ratio in both subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale numerical columns based ONLY on training metrics to avoid data leakage
numerical_cols = ['purchase_value', 'age', 'time_since_signup', 'user_count_per_device', 'user_count_per_ip']

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

In [ ]:
print(f"Original training class distribution: {np.bincount(y_train)}")

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Resample ONLY the training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled training class distribution: {np.bincount(y_train_resampled)}")